In [ ]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]

dataset = dataset.map(
    lambda x: {
        "combined_text": x["prompt"] + " " + x["A"]
    }
)

print(len(dataset[51]["combined_text"]))

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

In [ ]:
print(tokenizer.convert_tokens_to_ids("[SEP]"))

In [ ]:
encodings = tokenizer(
    list(dataset["prompt"]),   # Convert to a Python list
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("bert-base-uncased")

print("Hidden Size:", config.hidden_size)
print("Attention Heads:", config.num_attention_heads)
print("Head Dimension:", config.hidden_size // config.num_attention_heads)

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")

# Prompt from row 0
prompt = dataset[0]["prompt"]

# Default tokenization (no manual padding/truncation)
inputs = tokenizer(prompt, return_tensors="pt")

# Forward pass
outputs = model(**inputs)

print("Input shape:", inputs["input_ids"].shape)
print("Last hidden state shape:", outputs.last_hidden_state.shape)

In [ ]:
print(round(outputs.last_hidden_state[0, 0, :5].sum().item(), 4))

In [ ]:
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    attn_implementation="eager"
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(
        **inputs,
        output_attentions=True
    )

print(outputs.attentions is None)
print(len(outputs.attentions))

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(tokens)

fusion_idx = tokens.index("fusion")

attention = outputs.attentions[-1][0, 0]

weight = attention[0, fusion_idx].item()

print(round(weight, 4))

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Row ID 0 (zero-indexed)
prompt = dataset[0]["prompt"]
option_b = dataset[0]["B"]

# Generate embeddings
prompt_emb = model.encode(prompt, convert_to_tensor=True)
option_emb = model.encode(option_b, convert_to_tensor=True)

# Cosine similarity
score = util.cos_sim(prompt_emb, option_emb)

print(round(score.item(), 4))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import numpy as np

OPTIONS = ["A","B","C","D","E"]

# ----------------------------
# TF-IDF vocabulary
# ----------------------------
combined = []

for row in dataset:
    combined.append(
        " ".join([
            row["prompt"],
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"]
        ])
    )

tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(combined)

# ----------------------------
# MiniLM model
# ----------------------------
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# ----------------------------
# MAP@3 helper
# ----------------------------
def ap3(actual, preds):
    if actual in preds[:3]:
        return 1.0/(preds.index(actual)+1)
    return 0.0

tfidf_scores = []
minilm_scores = []

tfidf_top3 = []
minilm_top3 = []

# ----------------------------
# Evaluate every question
# ----------------------------
for row in dataset:

    prompt = row["prompt"]

    # ---------- TFIDF ----------
    prompt_vec = tfidf.transform([prompt])

    sims = {}

    for opt in OPTIONS:

        option_vec = tfidf.transform([row[opt]])

        sims[opt] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    ranked = sorted(
        sims,
        key=sims.get,
        reverse=True
    )

    tfidf_top3.append(ranked[:3])

    tfidf_scores.append(
        ap3(row["answer"], ranked)
    )

    # ---------- MiniLM ----------

    prompt_emb = model.encode(
        prompt,
        convert_to_tensor=True
    )

    sims = {}

    for opt in OPTIONS:

        option_emb = model.encode(
            row[opt],
            convert_to_tensor=True
        )

        sims[opt] = util.cos_sim(
            prompt_emb,
            option_emb
        ).item()

    ranked = sorted(
        sims,
        key=sims.get,
        reverse=True
    )

    minilm_top3.append(ranked[:3])

    minilm_scores.append(
        ap3(row["answer"], ranked)
    )

# ----------------------------
# Final MAP@3
# ----------------------------

tfidf_map3 = np.mean(tfidf_scores)
minilm_map3 = np.mean(minilm_scores)

print("TFIDF MAP@3 :", round(tfidf_map3,6))
print("MiniLM MAP@3:", round(minilm_map3,6))

# ----------------------------
# Improvement count
# ----------------------------

count = 0

for i,row in enumerate(dataset):

    answer = row["answer"]

    if (
        answer not in tfidf_top3[i]
        and
        answer in minilm_top3[i]
    ):
        count += 1

print("Improvement Count:", count)

In [ ]:
from transformers import pipeline

# Zero-shot pipeline
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Row index 1 (zero-indexed)
row = dataset[1]

sequence = row["prompt"]
candidate_labels = [
    row["A"],
    row["B"],
    row["C"]
]

result = classifier(
    sequence,
    candidate_labels
)

print("Labels :", result["labels"])
print("Scores :", result["scores"])
print("Top Probability:", round(result["scores"][0], 4))

In [ ]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row = dataset[1]

sequence = row["prompt"]
labels = [row["A"], row["B"], row["C"]]

# Softmax (default)
softmax_result = classifier(
    sequence,
    candidate_labels=labels
)

softmax_sum = sum(softmax_result["scores"])

# Independent Sigmoids
sigmoid_result = classifier(
    sequence,
    candidate_labels=labels,
    multi_label=True
)

sigmoid_sum = sum(sigmoid_result["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print("Softmax scores :", softmax_result["scores"])
print("Softmax sum    :", softmax_sum)

print("Sigmoid scores :", sigmoid_result["scores"])
print("Sigmoid sum    :", sigmoid_sum)

print("Difference     :", round(difference, 4))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row = dataset[0]

prompt = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} "
    f"or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)